# Solicitações (Silver)

Transforma `tb_solicitacoes` da Bronze em uma linha por solicitação, com tipos, marcos do ciclo de vida e classificação de desfecho. O histórico `logStatus` é preservado em formato longo.

## Fluxo dos dados

* Origem: `cat_mobilidade_urbana_bronze.db_frota.tb_solicitacoes`
* Destinos: `cat_mobilidade_urbana_silver.db_frota.tb_solicitacoes` e `cat_mobilidade_urbana_silver.db_frota.tb_solicitacoes_status`

## Tratamentos e evidências (perfil de 14/10/2026)

1. **Deduplicação por `codigo` (RN003):** a extração tem 451.386 documentos e 451.386 códigos distintos; a regra fica explícita para reprocessamentos.
2. **Tipagem temporal:** todos os carimbos têm offset `-03:00`; são convertidos para `timestamp` e a data/hora local usa `America/Sao_Paulo`.
3. **Marcos do ciclo de vida:** `dT_ACEITE_MOTORISTA` coincide com o primeiro `CondutorACaminho` e `dT_INICIO_REALIZADA` com o primeiro `CorridaIniciada` (mediana e p99 de diferença iguais a 0 s). Usa-se o campo do documento e, na falta, o log.
4. **Base temporal da demanda:** em `TempoReal`, `dT_REGISTRO` coincide com o primeiro `AguardandoCondutor` (mediana 0 s; hora local diferente em 1,5% dos casos). Ele é registrado como **entrada na fila de despacho**, proxy de criação ainda não homologada pela empresa (RN023).
5. **Desfecho:** o status final é classificado sem atribuir causa. `NaoAtendida` e `LimiteEsperaAtingido` não são tratados como cancelamento nem como falta de carro comprovada (RN004).

In [0]:
# No Databricks `spark` já existe; no VS Code o Databricks Connect cria uma sessão serverless.
try:
    spark  # noqa: F821
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless(True).getOrCreate()

from pyspark.sql import Window as W
from pyspark.sql import functions as F

# Todos os carimbos da fonte têm offset -03:00; datas e horas locais usam o fuso informado pelo proponente.
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

BRONZE = "cat_mobilidade_urbana_bronze.db_frota"
SILVER = "cat_mobilidade_urbana_silver.db_frota"
GOLD = "cat_mobilidade_urbana_gold.db_frota"


def segundos_entre(inicio, fim):
    """Diferença em segundos (com fração) entre dois timestamps."""
    return F.col(fim).cast("double") - F.col(inicio).cast("double")


def haversine_km(lat1, lon1, lat2, lon2):
    """Mesmo Haversine de src/tcc_frota/distance.py (R = 6371,0088 km).

    Linha reta entre dois pontos: estimativa geodésica, nunca rodagem observada (RN012).
    """
    la1, lo1, la2, lo2 = (F.radians(F.col(c)) for c in (lat1, lon1, lat2, lon2))
    h = F.pow(F.sin((la2 - la1) / 2), 2) + F.cos(la1) * F.cos(la2) * F.pow(F.sin((lo2 - lo1) / 2), 2)
    return F.lit(6371.0088) * 2 * F.asin(F.least(F.lit(1.0), F.sqrt(h)))


def pseudonimo(coluna):
    """Pseudônimo determinístico (SHA-256 truncado) para tabelas Gold; não é anonimização."""
    return F.substring(F.sha2(F.concat(F.lit("tcc-frota|"), F.col(coluna).cast("string")), 256), 1, 16)


def gravar(df, tabela, comentario):
    """Sobrescreve a tabela Delta de forma idempotente e registra o comentário."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

## 1. Leitura e tipagem

In [0]:
bronze = spark.table(f"{BRONZE}.tb_solicitacoes")

tipadas = bronze.select(
    F.col("codigo").cast("bigint").alias("id_solicitacao"),
    F.col("cD_REGIAO").cast("int").alias("id_regiao"),
    F.col("cD_MOTORISTA").cast("bigint").alias("id_motorista"),
    F.col("cD_VEICULO").cast("bigint").alias("id_veiculo"),
    F.col("tX_TP_SOLICITACAO").alias("tipo_solicitacao"),
    F.col("tX_STATUS").alias("status_final"),
    F.to_timestamp("dT_REGISTRO").alias("ts_registro"),
    F.to_timestamp("dT_ACEITE_MOTORISTA").alias("ts_aceite_doc"),
    F.to_timestamp("dT_CHEGADA_MOTORISTA").alias("ts_chegada_doc"),
    F.to_timestamp("dT_INICIO_REALIZADA").alias("ts_inicio_doc"),
    F.to_timestamp("dT_FIM_REALIZADA").alias("ts_fim_doc"),
    F.col("logStatus").alias("log_status"),
)

## 2. Deduplicação por solicitação

Mantém o documento com `dT_REGISTRO` mais recente e, em empate, o de histórico mais longo.

In [0]:
ordem = W.partitionBy("id_solicitacao").orderBy(
    F.col("ts_registro").desc_nulls_last(), F.size("log_status").desc()
)
dedup = (
    tipadas.where(F.col("id_solicitacao").isNotNull())
    .withColumn("_rn", F.row_number().over(ordem))
    .where("_rn = 1")
    .drop("_rn")
)
n_bronze, n_dedup = bronze.count(), dedup.count()
print(f"Bronze: {n_bronze:,} | Silver após dedup: {n_dedup:,} | removidos: {n_bronze - n_dedup:,}")

## 3. Histórico de status em formato longo

In [0]:
status = (
    dedup.select("id_solicitacao", F.posexplode("log_status").alias("seq_evento", "ev"))
    .select(
        "id_solicitacao",
        "seq_evento",
        F.col("ev.tX_STATUS").alias("status"),
        F.to_timestamp("ev.dT_REGISTRO").alias("ts_evento"),
    )
    .dropDuplicates(["id_solicitacao", "status", "ts_evento"])
    .withColumn("data_local", F.to_date("ts_evento"))
    .withColumn("_ts_processamento", F.current_timestamp())
)


def primeiro(nome_status):
    return F.min(F.when(F.col("status") == nome_status, F.col("ts_evento")))


marcos = status.groupBy("id_solicitacao").agg(
    primeiro("AguardandoCondutor").alias("ts_log_aguardando"),
    primeiro("CondutorACaminho").alias("ts_log_a_caminho"),
    primeiro("CondutorChegou").alias("ts_log_chegou"),
    primeiro("CorridaIniciada").alias("ts_log_iniciada"),
    primeiro("CorridaEncerrada").alias("ts_log_encerrada"),
    F.max("ts_evento").alias("ts_ultimo_evento"),
    F.count("*").alias("qtd_eventos_status"),
    F.sum(F.when(F.col("status") == "CondutorACaminho", 1).otherwise(0)).alias("qtd_aceites"),
    F.max(F.col("status") == "AguardandoCondutorEmOutraViagem").alias("fl_aguardou_condutor_em_outra_viagem"),
)

## 4. Classificação do desfecho

`grupo_desfecho` só agrupa nomes de status. A causa do não atendimento continua pendente (RN004); cancelamento não é sinônimo de ausência de veículo.

In [0]:
desfechos = spark.createDataFrame(
    [
        ("CorridaEncerrada", "ATENDIDA", "ATENDIDA"),
        ("NaoAtendida", "NAO_ATENDIDA", "NAO_ATENDIDA_OU_EXPIRADA"),
        ("LimiteEsperaAtingido", "LIMITE_ESPERA_ATINGIDO", "NAO_ATENDIDA_OU_EXPIRADA"),
        ("CorridaCanceladaSolicitante", "CANCELADA_SOLICITANTE", "CANCELADA"),
        ("CorridaCanceladaSolicitantePosAceite", "CANCELADA_SOLICITANTE_POS_ACEITE", "CANCELADA"),
        ("CorridaCanceladaMotoristaPosAceite", "CANCELADA_MOTORISTA_POS_ACEITE", "CANCELADA"),
        ("CorridaCanceladaOperador", "CANCELADA_OPERADOR", "CANCELADA"),
        ("CorridaCanceladaAgendamento", "CANCELADA_AGENDAMENTO", "CANCELADA"),
        ("PagamentoRejeitado", "PAGAMENTO_REJEITADO", "OUTRO_DESFECHO"),
        ("Retracamento", "RETRACAMENTO", "OUTRO_DESFECHO"),
        ("CondutorACaminho", "EM_ANDAMENTO_NA_EXTRACAO", "EM_ANDAMENTO"),
        ("CondutorChegou", "EM_ANDAMENTO_NA_EXTRACAO", "EM_ANDAMENTO"),
        ("CorridaIniciada", "EM_ANDAMENTO_NA_EXTRACAO", "EM_ANDAMENTO"),
    ],
    "status_final string, categoria_desfecho string, grupo_desfecho string",
)

## 5. Montagem da Silver de solicitações

In [0]:
s = (
    dedup.drop("log_status")
    .join(marcos, "id_solicitacao", "left")
    .join(F.broadcast(desfechos), "status_final", "left")
    .withColumn("categoria_desfecho", F.coalesce("categoria_desfecho", F.lit("NAO_MAPEADO")))
    .withColumn("grupo_desfecho", F.coalesce("grupo_desfecho", F.lit("NAO_MAPEADO")))
    .withColumn("ts_entrada_fila", F.col("ts_log_aguardando"))
    .withColumn("ts_aceite", F.coalesce("ts_aceite_doc", "ts_log_a_caminho"))
    .withColumn("ts_chegada", F.coalesce("ts_chegada_doc", "ts_log_chegou"))
    .withColumn("ts_inicio", F.coalesce("ts_inicio_doc", "ts_log_iniciada"))
    .withColumn("ts_fim", F.coalesce("ts_fim_doc", "ts_log_encerrada"))
    .withColumn("data_local", F.to_date("ts_registro"))
    .withColumn("hora_local", F.hour("ts_registro"))
    .withColumn("dia_semana_iso", ((F.dayofweek("ts_registro") + 5) % 7) + 1)
    .withColumn("fl_fim_de_semana", F.col("dia_semana_iso") >= 6)
    .withColumn("seg_registro_ate_aceite", segundos_entre("ts_registro", "ts_aceite"))
    .withColumn("seg_pickup", segundos_entre("ts_aceite", "ts_chegada"))
    .withColumn("seg_espera_embarque", segundos_entre("ts_chegada", "ts_inicio"))
    .withColumn("seg_viagem", segundos_entre("ts_inicio", "ts_fim"))
    .withColumn("fl_atendida", F.col("categoria_desfecho") == "ATENDIDA")
    .withColumn("fl_sem_log_status", F.col("qtd_eventos_status").isNull())
    .withColumn(
        "fl_tempos_ordenados",
        F.when(
            F.col("fl_atendida"),
            F.coalesce(
                (F.col("ts_registro") <= F.col("ts_aceite"))
                & (F.col("ts_aceite") <= F.col("ts_chegada"))
                & (F.col("ts_chegada") <= F.col("ts_inicio"))
                & (F.col("ts_inicio") < F.col("ts_fim")),
                F.lit(False),
            ),
        ),
    )
    .withColumn("base_temporal_demanda", F.lit("dT_REGISTRO_entrada_fila_proxy_nao_homologada"))
    .withColumn("_fonte", F.lit(f"{BRONZE}.tb_solicitacoes"))
    .withColumn("_ts_processamento", F.current_timestamp())
)

colunas = [
    "id_solicitacao", "id_regiao", "tipo_solicitacao", "status_final", "categoria_desfecho", "grupo_desfecho",
    "fl_atendida", "id_motorista", "id_veiculo",
    "ts_registro", "ts_entrada_fila", "ts_aceite", "ts_chegada", "ts_inicio", "ts_fim", "ts_ultimo_evento",
    "data_local", "hora_local", "dia_semana_iso", "fl_fim_de_semana",
    "seg_registro_ate_aceite", "seg_pickup", "seg_espera_embarque", "seg_viagem",
    "qtd_eventos_status", "qtd_aceites", "fl_aguardou_condutor_em_outra_viagem",
    "fl_sem_log_status", "fl_tempos_ordenados", "base_temporal_demanda", "_fonte", "_ts_processamento",
]
solicitacoes = s.select(*colunas)

## 6. Persistência

In [0]:
gravar(
    solicitacoes,
    f"{SILVER}.tb_solicitacoes",
    "Uma linha por solicitacao (dedup por codigo), tempos tipados e desfecho classificado sem atribuir causa. "
    "ts_registro e proxy de entrada na fila, nao criacao homologada (RN023).",
)
gravar(
    status,
    f"{SILVER}.tb_solicitacoes_status",
    "Historico logStatus em formato longo, um evento por linha, deduplicado por solicitacao, status e instante.",
)

## 7. Conferência agregada

In [0]:
display_df = (
    spark.table(f"{SILVER}.tb_solicitacoes")
    .groupBy("tipo_solicitacao", "grupo_desfecho", "categoria_desfecho")
    .agg(F.count("*").alias("qtd"), F.avg(F.col("fl_tempos_ordenados").cast("int")).alias("pct_tempos_ordenados"))
    .orderBy(F.desc("qtd"))
)
for linha in display_df.collect():
    print(linha.asDict())